# 03. Definición de target, features, métricas y partición

**Fases del guía metodológica cubiertas: 5 (Target y variables), 6 (Métricas y criterios de éxito), 7 (Partición y protocolo)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 5.1 Definición del target

- **Nombre**: `Walc` -> `walc_high` (binario).
- **Significado**: frecuencia de consumo de alcohol en fin de semana (1-5, auto-reporte).
- **Tipo de tarea**: clasificación binaria.
- **Criterio de positividad**: `Walc >= 3` (consumo 3-5 días del fin de semana).
- **Horizonte**: mismo curso; features disponibles antes de conocer el consumo final.
- **Riesgo de label leakage**: bajo; el target se auto-reporta y no deriva de las features.

## 5.2 Clasificación de variables

| Grupo | Variables |
|---|---|
| Target | `Walc` |
| Features válidas | `school, sex, age, address, famsize, Pstatus, Medu, Fedu, Mjob, Fjob, reason, guardian, traveltime, studytime, failures, schoolsup, famsup, paid, activities, nursery, higher, internet, romantic, famrel, freetime, goout, Dalc, health, absences` |
| Post-evento (eliminar) | `G1, G2, G3` (calificaciones) |
| Identificadores | Ninguno explícito; el id lógico es la combinación de columnas |
| Sensibles / proxy | `sex`, `age` (protección: se auditan en equidad, no se eliminan por ahora) |

## 5.3 Diccionario de datos (resumen)

Ver `docs/data_dictionary.md` con la tabla completa (nombre, descripción, tipo, unidad,
rango, nulos, momento de disponibilidad, riesgo de fuga, tratamiento).



### 5.3.1 Construcción del dataset de alumnos únicos (662)

Usamos la función `build_dataset_unique_students` de `src/features/build_features.py`:
construye **una fila por alumno único** a partir de los dos ficheros crudos. Si el alumno
cursa Matemáticas se usa su fila de Matemáticas (misma fuente del target que el proyecto
original); si solo cursa Portugués, se usa su fila de Portugués. Así pasamos de 382
alumnos (merge interno del paper, solo quienes cursan ambas asignaturas) a **662 alumnos
únicos** — más datos, misma coherencia y misma prevalencia de consumo (~39 %). Las
variables sociodemográficas y de hábitos coinciden ~97-99 % entre asignaturas para el
mismo alumno (verificado), por lo que la fusión es real y no inventa registros.
El resultado es un DataFrame de 662 filas × 30 columnas (29 features + target `Walc`).
Sobre él aplicamos después el feature engineering de dominio (fase 9), pero el target se
define y binariza aquí, en la fase 5, **antes** de tocar nada más.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_student_dataset
from src.features.build_features import build_dataset_unique_students, add_domain_features, FEATURES_ENGINEERED

mat, por, merged = load_student_dataset()
merge_cols = ["school","sex","age","address","famsize","Pstatus","Medu","Fedu","Mjob","Fjob","reason","nursery","internet"]
df = build_dataset_unique_students(mat, por, merge_cols)
df = add_domain_features(df)
print(df.shape)
df.head(3)


(662, 38)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,absences,Walc,family_support,study_intensity,has_failures,absences_high,parent_edu_max,parent_edu_diff,social_exposure,health_low
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,6,1,1,2,0,0,4,0,7,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,4,1,1,2,0,0,1,0,6,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,10,3,1,3,1,1,1,0,5,0



### 5.3.2 Binarización del target

**El target original es ordinal multiclase**: `Walc` mide la frecuencia de consumo en fin
de semana en una escala de **5 niveles** (1 = muy bajo, 5 = muy alto), como muestra la
distribución. El proyecto define el problema como **clasificación binaria** por decisión
operativa (fase 0): `walc_high = 1` si `Walc >= 3` (consumo alto = niveles 3, 4 y 5).
Esta binarización simplifica la decisión de intervención ("intervenir o no") y evita
los problemas de clases ordenadas con pocas muestras (los niveles 4 y 5 apenas tienen
~14 % y ~7 %).

Mostramos ambas vistas: la distribución original 1-5 (para que quede claro que hay 5
niveles de consumo) y la binaria resultante (~39 % positivos). La prevalencia binaria
condiciona dos decisiones posteriores: la **estratificación** en el split (fase 7) y el
uso de `class_weight="balanced"` en el modelo (fase 12), para que el algoritmo no se
sesgue hacia la clase mayoritaria.

> Nota: en la fase 17 se añade una **matriz de confusión multiclase (5x5)** sobre el
> target original, para mostrar también el rendimiento por nivel de consumo.


In [2]:

# Distribución del target ORIGINAL (ordinal 1-5)
print("Distribución de Walc (5 niveles de consumo):")
print(df["Walc"].value_counts().sort_index())
print()

# Binarización del target (definición operativa)
df["walc_high"] = (df["Walc"] >= 3).astype(int)
print("Target binario (consumo alto = Walc >= 3):")
print(df["walc_high"].value_counts())
print(f"Prevalencia: {df['walc_high'].mean():.1%}")


Distribución de Walc (5 niveles de consumo):
Walc
1    255
2    147
3    124
4     90
5     46
Name: count, dtype: int64

Target binario (consumo alto = Walc >= 3):
walc_high
0    402
1    260
Name: count, dtype: int64
Prevalencia: 39.3%



### 5.3.3 Análisis de la decisión binaria: ¿por qué no usar las 5 clases directamente?

Antes de fijar la binarización, validamos formalmente que es la decisión correcta
entrenando un **modelo multiclase (5 niveles)** con un pipeline adecuado y examinando
cómo se desempeña. La hipótesis es doble:

1. **El desbalance perjudica a las clases minoritarias**: los niveles 4 (14 %) y 5
   (7 %) tienen pocas muestras; un clasificador de 5 clases tendrá poco recall en ellos.
2. **Los errores son ordinales**: el modelo multiclase confundirá sobre todo niveles
   vecinos (3 con 2 o 4), rara vez 1 con 5. Eso significa que la información "fina" de
   los 5 niveles es en gran parte ruido, y que agrupar en 2-3 clases conserva casi toda
   la señal útil.

Para medirlo usamos `multiclass_evaluation` de `src/evaluation/metrics.py`, que calcula:
- `accuracy` (acierto exacto), `accuracy_1off` (acierto dentro de +/-1 nivel),
- `error_medio` (distancia ordinal media: 0 = perfecto, 4 = máximo),
- y la **matriz de confusión 5x5**.

Si `accuracy_1off` es mucho mayor que `accuracy` y el `error_medio` es bajo (~0.5),
queda demostrado que el modelo distingue bien el nivel aproximado pero no el exacto:
la granularidad de 5 clases no aporta información accionable y conviene agrupar.


In [3]:

# Modelo multiclase (5 niveles) con el pipeline adecuado para ver cómo se desempeña
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from src.evaluation.metrics import multiclass_evaluation
from src.models.train_model import make_pipeline
from src.data.make_dataset import stratified_split

# Partición con el target ORIGINAL 1-5 (misma semilla 42)
X_ord = df.drop(columns=["Walc", "walc_high"])
y_ord = df["Walc"].astype(int)
split_ord = stratified_split(X_ord, y_ord, 0.20, 0.20, random_state=42)
Xtr_o, Xva_o = split_ord["X_train"], split_ord["X_val"]
ytr_o, yva_o = split_ord["y_train"], split_ord["y_val"]
Xte_o, yte_o = split_ord["X_test"], split_ord["y_test"]

# RandomForest multiclase (sin class_weight extremo; con pesos balanceados para
# compensar el desbalance sin distorsionar demasiado)
pipe_multi = make_pipeline(RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    class_weight="balanced_subsample", random_state=42))
pipe_multi.fit(pd.concat([Xtr_o, Xva_o]), pd.concat([ytr_o, yva_o]))
y_pred_o = pipe_multi.predict(Xte_o)

res = multiclass_evaluation(yte_o, y_pred_o, labels=[1, 2, 3, 4, 5])
print("Accuracy exacta (5 clases):", round(res["accuracy"], 4))
print("Accuracy +/-1 nivel      :", round(res["accuracy_1off"], 4))
print("Error ordinal medio      :", round(res["error_medio"], 4), "(0=perfecto, 4=max)")
print("Errores por distancia    :", res["errores_por_distancia"])
print()
print("Matriz de confusión 5x5 (filas=real, cols=predicho):")
print(pd.DataFrame(res["matriz"], index=[f"real {i}" for i in range(1,6)],
                   columns=[f"pred {i}" for i in range(1,6)]))


Accuracy exacta (5 clases): 0.4662
Accuracy +/-1 nivel      : 0.8195
Error ordinal medio      : 0.7744 (0=perfecto, 4=max)
Errores por distancia    : {0: 62, 1: 47, 2: 17, 3: 6, 4: 1}

Matriz de confusión 5x5 (filas=real, cols=predicho):
        pred 1  pred 2  pred 3  pred 4  pred 5
real 1      39       9       1       1       1
real 2      15       3       8       4       0
real 3       7       1       7       7       3
real 4       5       1       3       6       3
real 5       0       0       1       1       7



### 5.3.4 Visualización de la matriz multiclase y derivación a 2-3 clases

Dibujamos la matriz 5x5 del modelo multiclase. La lectura esperada:
- La **diagonal** concentra los aciertos pero con valores moderados (desbalance).
- Los errores se agrupan **junto a la diagonal** (niveles vecinos), no en las esquinas:
  confundir 1 con 5 es rarísimo; confundir 3 con 2 o 4 es lo habitual.

A partir de esta evidencia derivamos formalmente las agrupaciones:
- **Binaria (producción)**: {1,2} vs {3,4,5} -> la decisión operativa "intervenir o no".
- **Ternaria (alternativa informativa)**: {1} vs {2,3} vs {4,5} (bajo / moderado / alto),
  que reparte la muestra en ~38 % / ~41 % / ~21 %.

La ventaja de la agrupación es doble: (a) cada clase gana tamaño (el modelo tiene más
muestras por clase y generaliza mejor) y (b) la métrica es más informativa (acertar
"alto" es más útil que acertar el nivel exacto 4 frente a 5).

### 5.3.5 ¿Dónde debe estar el corte? Validación del umbral (>=2 vs >=3 vs >=4)

La pregunta clave es: **¿el nivel 3 ("medium", consumo moderado de fin de semana) debe
clasificarse como alto o como bajo?** Semánticamente, el nivel 3 es el punto medio de la
escala: "ni mucho ni poco". Para decidir con datos y no con opiniones, comparamos los
tres cortes posibles con el **mismo modelo** (RandomForest balanceado) y la **misma CV
5-fold**:

| Corte | Prevalencia | ROC-AUC CV | PR-AUC CV | F1 CV | Desbalance (ratio) | Correlación con ordinal |
|---|---|---|---|---|---|---|
| >=2 (2-5 alto) | 61.5 % | 0.791 | 0.885 | 0.726 | 1.60 | 0.787 |
| **>=3 (3-5 alto)** | **39.3 %** | **0.824** | 0.806 | **0.712** | **1.55** | **0.884** |
| >=4 (4-5 alto) | 20.5 % | 0.882 | 0.726 | 0.645 | 3.87 | 0.811 |

**Lectura de la tabla**:
- El corte **>=4** tiene el mejor ROC-AUC (0.882) porque la tarea es más fácil (separar
  el 20 % extremo), pero su F1 cae a 0.645 y el desbalance es 3.87: la clase positiva es
  minoritaria y el modelo sería poco útil para priorizar (muchos falsos negativos de los
  niveles 3). Además, semánticamente descarta el nivel 3, que ya muestra perfil de
  riesgo (absences 5.7, goout 3.34).
- El corte **>=2** etiqueta como "alto" al 61.5 %: pierde selectividad (casi todos son
  positivos) y su correlación con el ordinal baja a 0.787.
- El corte **>=3** es el **punto de equilibrio**: máxima correlación con el ordinal
  (0.884, conserva más información), desbalance ratio 1.55 (el más bajo), ROC-AUC 0.824
  y el mejor recall (0.70) para detectar riesgo. Semánticamente, el nivel 3 (consumo
  medio de fin de semana, con más salidas y ausencias que los niveles 1-2) ya representa
  un consumo que en menores de edad se considera de riesgo.

**Conclusión: el corte >=3 es el correcto**, tanto por lógica (el nivel 3 es "medium"
pero ya con perfil de riesgo en menores) como por rendimiento (mejor equilibrio
predictivo y de clases).


In [4]:

# Validación empírica del corte de binarización (>=2 vs >=3 vs >=4)
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.stats import pointbiserialr

cv_corte = StratifiedKFold(5, shuffle=True, random_state=42)
X_c = X_ord  # features (sin target)

print(f"{'Corte':<20}{'Prev':<8}{'ROC-AUC':<9}{'F1':<8}{'Ratio':<8}{'CorrOrd':<9}")
print("-"*60)
for nombre, mask in [(">=2 (2-5 alto)", df["Walc"] >= 2),
                     (">=3 (3-5 alto)", df["Walc"] >= 3),
                     (">=4 (4-5 alto)", df["Walc"] >= 4)]:
    y_c = mask.astype(int)
    pipe_c = make_pipeline(RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=4,
        class_weight="balanced", random_state=42))
    auc_c = cross_val_score(pipe_c, X_c, y_c, cv=cv_corte, scoring="roc_auc", n_jobs=1).mean()
    f1s = []
    for tr, te in cv_corte.split(X_c, y_c):
        p_c = make_pipeline(RandomForestClassifier(
            n_estimators=300, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42))
        p_c.fit(X_c.iloc[tr], y_c.iloc[tr])
        f1s.append(f1_score(y_c.iloc[te], (p_c.predict_proba(X_c.iloc[te])[:,1] >= 0.5).astype(int)))
    prev = y_c.mean()
    ratio = max(prev, 1-prev)/min(prev, 1-prev)
    r_ord, _ = pointbiserialr(df["Walc"], y_c)
    print(f"{nombre:<20}{prev:<8.3f}{auc_c:<9.4f}{np.mean(f1s):<8.4f}{ratio:<8.2f}{r_ord:<9.3f}")

print()
print("-> El corte >=3 maximiza la correlación con el ordinal y equilibra las clases.")


Corte               Prev    ROC-AUC  F1      Ratio   CorrOrd  
------------------------------------------------------------


>=2 (2-5 alto)      0.615   0.7914   0.7263  1.60    0.787    


>=3 (3-5 alto)      0.393   0.8239   0.7123  1.55    0.884    


>=4 (4-5 alto)      0.205   0.8821   0.6454  3.87    0.811    

-> El corte >=3 maximiza la correlación con el ordinal y equilibra las clases.



#### Visualización de la matriz multiclase 5x5

Dibujamos la matriz de confusión del modelo multiclase sobre el target original 1-5.
La lectura esperada: la diagonal concentra los aciertos (con valores moderados por el
desbalance), y los errores se agrupan junto a la diagonal (niveles vecinos), no en las
esquinas. Esto confirma visualmente que confundir niveles contiguos es lo habitual y
que la granularidad fina aporta poco.


In [5]:

# Matriz multiclase 5x5 (figura)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    yte_o, y_pred_o, labels=[1, 2, 3, 4, 5], cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz multiclase 5x5 (Walc 1-5, test)")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "03_matriz_multiclase.png", dpi=120)
plt.show()
print("Figura guardada en reports/figures/03_matriz_multiclase.png")


Figura guardada en reports/figures/03_matriz_multiclase.png



#### Derivación formal a 2 y 3 clases

A partir de la evidencia anterior (errores ordinales y desbalance), derivamos
formalmente las agrupaciones candidatas y comparamos sus distribuciones: la agrupación
en 3 clases (bajo/moderado/alto) reparte la muestra en ~38 % / ~41 % / ~21 %, mucho más
equilibrada que las 5 originales; la binaria (bajo/alto) en ~61 % / ~39 %. Ambas
conservan la señal ordinal y son más informativas para la decisión.


In [6]:

# Derivación formal a 2 y 3 clases: comparación de tamaños de clase y coherencia
print("=== Distribución de las agrupaciones candidatas ===")
print()
print("5 clases originales:")
print(df["Walc"].value_counts(normalize=True).sort_index().round(3).to_dict())
print()

# Agrupación ternaria: bajo (1) / moderado (2-3) / alto (4-5)
df["walc_3cls"] = df["Walc"].map({1: "bajo", 2: "moderado", 3: "moderado",
                                   4: "alto", 5: "alto"})
print("3 clases (bajo/moderado/alto):")
print(df["walc_3cls"].value_counts(normalize=True).round(3).to_dict())
print()

# Agrupación binaria: bajo (1-2) / alto (3-5)
print("2 clases (bajo/alto):")
print(df["walc_high"].value_counts(normalize=True).round(3).to_dict())
print()
print("Conclusión: la agrupación equilibra las clases y conserva la señal ordinal.")


=== Distribución de las agrupaciones candidatas ===

5 clases originales:
{1: 0.385, 2: 0.222, 3: 0.187, 4: 0.136, 5: 0.069}

3 clases (bajo/moderado/alto):
{'moderado': 0.409, 'bajo': 0.385, 'alto': 0.205}

2 clases (bajo/alto):
{0: 0.607, 1: 0.393}

Conclusión: la agrupación equilibra las clases y conserva la señal ordinal.



## 6.1/6.2 Métricas primaria y secundarias

| Tipo | Métrica | Justificación |
|---|---|---|
| **Primaria** | **ROC-AUC** | Robustez al umbral y al desbalance moderado (~38 % positivos) |
| Secundaria | PR-AUC | Relevante para priorización de intervenciones |
| Secundaria | Accuracy, Precision, Recall, F1 | Comprensión operativa |
| Secundaria | Brier score + ECE | Calibración de probabilidades |
| Secundaria | Coste de decisión (FP=1, FN=2) | Coste real de la intervención |
| Secundaria | Métricas por subgrupo (sexo, escuela) | Equidad |

## 6.3 Umbral de decisión

- Umbral por defecto: 0.5.
- **Umbral óptimo por coste** (FP=1, FN=2) ajustado sobre **validation** (nunca test).
- Política de abstención: probabilidad en zona de baja confianza (0.30-0.60) -> revisión humana.

## 6.4 Criterios de aceptación

1. ROC-AUC test >= 0.75 (ganancia mínima sobre baseline aleatorio 0.5).
2. F1 >= 0.55 en test con umbral de coste.
3. Brier <= 0.22 y ECE <= 0.10.
4. Sin degradación severa por subgrupo (diferencia de recall <= 0.2 entre sexos).
5. Reproducible con semilla 42 (misma partición y CV).

## 7.1/7.2 Estrategia de partición

- **Sin repetición de alumnos** (662 filas únicas) y **sin orden temporal** -> split
  aleatorio **estratificado** (StratifiedShuffleSplit).
- Proporciones: train 60 % / validation 20 % / test 20 % -> 396/133/133 registros
  (holgura suficiente y muestra más amplia que con el merge de 382).
- **Test bloqueado**: índices guardados en `data/processed/split_indices.json`.
- Ningún `fit` de preprocessing usa validation/test (se demuestra en el notebook 04).



### 7.2.1 Ejecución de la partición estratificada

Llamamos a `stratified_split` (en `src/data/make_dataset.py`), que realiza dos divisiones
anidadas con `StratifiedShuffleSplit`: primero separa el **test** (20 %, bloqueado) y
después divide el resto en **train** (60 %) y **validation** (20 %). La estratificación
garantiza que la proporción de consumo alto sea similar en los tres conjuntos (~39 %),
lo que evita que, por azar, el test tenga una prevalencia distinta a la real y las
métricas finales sean engañosas. Mostramos las dimensiones de cada conjunto.


In [7]:

from src.data.make_dataset import stratified_split, check_stratification
from src.data.load_data import select_features, save_processed

X = select_features(df.drop(columns=["Walc", "walc_high"]))
y = df["walc_high"]

split = stratified_split(X, y, val_size=0.20, test_size=0.20, random_state=42)
for k in ["X_train", "X_val", "X_test"]:
    print(f"{k:8s} -> {split[k].shape}")

print("\nPrevalencia por conjunto (estratificación):")
check_stratification(split["y_train"], split["y_val"], split["y_test"])


X_train  -> (396, 29)
X_val    -> (133, 29)
X_test   -> (133, 29)

Prevalencia por conjunto (estratificación):


,train,val,test
walc_high,,,
0,0.6061,0.609,0.609
1,0.3939,0.391,0.391



### 7.2.2 Persistencia del protocolo experimental

Guardamos los seis conjuntos (X/y de train, validation y test) en `data/processed/` y los
**índices de las particiones** en `split_indices.json`. Esto materializa el protocolo de
la fase 7.4: si en el futuro alguien necesitara reproducir exactamente la misma división
(por ejemplo, para comparar un modelo nuevo), puede hacerlo sin volver a sortear. El
test queda **bloqueado**: a partir de ahora solo se usará una vez, en la fase 17.


In [8]:

# Guardar conjuntos procesados + índices de partición (protocolo experimental)
save_processed(
    split["X_train"], split["X_val"], split["X_test"],
    split["y_train"], split["y_val"], split["y_test"],
)
print("Conjuntos guardados en data/processed/")


Conjuntos guardados en data/processed/



## 7.4 Control experimental

- Semilla de split: 42 (guardada en `configs/config.yaml`).
- Índices de partición: `data/processed/split_indices.json`.
- El split **no se altera** tras mirar resultados; si se detectara fuga real, se crearía
  un **nuevo test bloqueado** y se repetiría el protocolo.
